In [1]:
import gym
import random
import numpy as np
import pandas as pd

env = gym.make('FrozenLake-v1')

alpha = 0.85
gamma = 0.90
epsilon = 0.8

num_episodes = 50000
num_timesteps = 1000


def random_policy():
    return env.action_space.sample()


env.action_space

v = {}

for s in range(env.observation_space.n):
    v[s] = 0.0


if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_


for i in range(num_episodes):
    s = env.reset()

    if isinstance(s, tuple):
        s = s[0]

    for t in range(num_timesteps):
        a = random_policy()

        step_result = env.step(a)

        s_, r, done = (
            step_result[0],
            step_result[1],
            step_result[2]
        )

        v[s] += alpha * (r + gamma * v[s_] - v[s])

        s = s_

        # a = a_

        if done:
            break


df = pd.DataFrame(
    list(v.items()),
    columns=['state', 'value']
)

df

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,state,value
0,0,0.001689
1,1,0.005745
2,2,0.002870
3,3,0.000229
4,4,0.000146
5,5,0.000000
6,6,0.018363
7,7,0.000000
8,8,0.000358
9,9,0.013966


In [2]:
def epsilon_greedy(state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    else:
        return max(
            list(range(env.action_space.n)),
            key=lambda x: Q[(state, x)]
        )


Q = {}

for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        Q[(s, a)] = 0.0


s = env.reset()

if isinstance(s, tuple):
    s = s[0]

a = epsilon_greedy(s, epsilon)

for t in range(num_timesteps):
    step_result = env.step(a)

    s_, r, terminated, truncated, _ = (
        step_result[0],
        step_result[1],
        step_result[2],
        step_result[3] if len(step_result) == 5 else False,
        step_result[-1]
    )

    done = terminated or truncated

    if isinstance(s_, tuple):
        s_ = s_[0]

    a_ = epsilon_greedy(s_, epsilon)

    Q[(s, a)] += alpha * (
        r + gamma * Q[(s_, a_)] - Q[(s, a)]
    )

    s = s_
    a = a_

    if done:
        break


df = pd.DataFrame(
    [
        (state, action, value)
        for (state, action), value in Q.items()
    ],
    columns=['state', 'action', 'value']
)

df

,state,action,value
0,0,0,0.0
1,0,1,0.0
2,0,2,0.0
3,0,3,0.0
4,1,0,0.0
...,...,...,...
59,14,3,0.0
60,15,0,0.0
61,15,1,0.0
62,15,2,0.0


In [3]:
import numpy as np

s = env.reset()

if isinstance(s, tuple):
    s = s[0]

for t in range(num_timesteps):
    a = epsilon_greedy(s, epsilon)

    step_result = env.step(a)
    s_, r, terminated, truncated, _ = (
        step_result[0],
        step_result[1],
        step_result[2],
        step_result[3] if len(step_result) == 5 else False,
        step_result[-1]
    )

    done = terminated or truncated

    a_ = np.argmax([
        Q[(s_, action)]
        for action in range(env.action_space.n)
    ])

    Q[(s, a)] += alpha * (
        r + gamma * Q[(s_, a_)] - Q[(s, a)]
    )

    s = s_

    if done:
        break

df = pd.DataFrame(
    [
        (state, action, value)
        for (state, action), value in Q.items()
    ],
    columns=['state', 'action', 'value']
)

df

,state,action,value
0,0,0,0.0
1,0,1,0.0
2,0,2,0.0
3,0,3,0.0
4,1,0,0.0
...,...,...,...
59,14,3,0.0
60,15,0,0.0
61,15,1,0.0
62,15,2,0.0


In [7]:
import random
import numpy as np
import gymnasium as gym

# Create Taxi environment without rendering
env = gym.make("Taxi-v4")

# Initialize Q-table
q = {}

for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        q[(s, a)] = 0.0


# Q-Learning update
def update_q_table(prev_state, action, reward, next_state, alpha, gamma):
    best_next_q = max(
        q[(next_state, a)]
        for a in range(env.action_space.n)
    )

    q[(prev_state, action)] += alpha * (
        reward + gamma * best_next_q - q[(prev_state, action)]
    )


# Epsilon-greedy policy
def epsilon_greedy_policy(state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()

    return max(
        range(env.action_space.n),
        key=lambda a: q[(state, a)]
    )


# Parameters
alpha = 0.4
gamma = 0.999
epsilon = 0.017
episodes = 8000


# Training
for i in range(episodes):
    total_reward = 0
    prev_state, info = env.reset()

    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = epsilon_greedy_policy(prev_state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        update_q_table(
            prev_state, action, reward, next_state, alpha, gamma
        )

        prev_state = next_state
        total_reward += reward

    if (i + 1) % 1000 == 0:
        print(f"Episode {i + 1}: Total reward = {total_reward}")

env.close()

print("\nTraining completed!")


# Test the trained agent and display the grid
test_env = gym.make("Taxi-v4", render_mode="ansi")

state, info = test_env.reset()

print("\nInitial Taxi Environment:")
print(test_env.render())

total_reward = 0
terminated = False
truncated = False
steps = 0

while not (terminated or truncated):
    action = max(
        range(test_env.action_space.n),
        key=lambda a: q[(state, a)]
    )

    state, reward, terminated, truncated, info = test_env.step(action)

    total_reward += reward
    steps += 1

    print(f"\nStep {steps}: Reward = {reward}")
    print(test_env.render())


print("\nTest Results")
print("Total reward:", total_reward)
print("Total steps:", steps)

test_env.close()

Episode 1000: Total reward = 6
Episode 2000: Total reward = 8
Episode 3000: Total reward = 8
Episode 4000: Total reward = 9
Episode 5000: Total reward = 7
Episode 6000: Total reward = 11
Episode 7000: Total reward = 6
Episode 8000: Total reward = 6

Training completed!

Initial Taxi Environment:
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+



Step 1: Reward = -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)


Step 2: Reward = -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)


Step 3: Reward = -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)


Step 4: Reward = -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Pickup)


Step 5: Reward = -1
+---------+
|R: | : :G|
|_: | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)


Step 6: Reward = -1
+---------+
|R: 